# Notebook Pelatihan Sentimen (Submission Dicoding)
Notebook ini berisi proses machine learning end-to-end dari data hasil scraping yang sudah tersedia, tanpa kode scraping.


## 1) Setup dan Load Dataset Hasil Scraping
Notebook ini membaca file data scraping data/raw/playstore_reviews_raw_latest.csv dan menjaga jumlah data tetap 10000 (5000 Gojek + 5000 Grab).


In [1]:
from pathlib import Path
import json
import re

import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

cwd = Path.cwd().resolve()
project_root = cwd.parent if cwd.name == "notebooks" else cwd
raw_path = project_root / "data" / "raw" / "playstore_reviews_raw_latest.csv"

raw_df = pd.read_csv(raw_path)
print("Project root:", project_root)
print("Raw data path:", raw_path)
print("Raw rows:", len(raw_df))
raw_df.head()


Project root: C:\PT InfantAI Teknologi Nusantara\shabi\shabi_v2
Raw data path: C:\PT InfantAI Teknologi Nusantara\shabi\shabi_v2\data\raw\playstore_reviews_raw_latest.csv
Raw rows: 10000


,date,username,content,score,thumbs_up_count,review_created_version,brand,app_id,source
0,2026-04-11 12:24:23,Raffa Fathan,sangat membantu terima kasih karya anak bangsa,5,0,NaN,gojek,com.gojek.app,google_play
1,2026-04-11 12:24:13,Muhammad zuki,Mau upgrade salah mulu photonya padahal uda be...,1,0,5.55.2,gojek,com.gojek.app,google_play
2,2026-04-11 12:21:35,Atikah Tikah,"mobil nya bersih wangi,dan pelayanannya ramah",4,0,5.55.2,gojek,com.gojek.app,google_play
3,2026-04-11 12:17:04,Dwi Sandarti,"cepat, nyaman dan aman",5,0,5.55.2,gojek,com.gojek.app,google_play
4,2026-04-11 11:50:29,TONI HERYAWAN JUARA GROUP,sangat membantu 👌🏼,5,0,5.55.2,gojek,com.gojek.app,google_play


In [2]:
brand_counts = raw_df["brand"].astype(str).str.lower().value_counts()

print("Total rows:", len(raw_df))
print("Brand distribution:")
print(brand_counts)

assert len(raw_df) == 10000, "Total data harus tetap 10000."
assert int(brand_counts.get("gojek", 0)) == 5000, "Data Gojek harus tetap 5000."
assert int(brand_counts.get("grab", 0)) == 5000, "Data Grab harus tetap 5000."

print("Validasi jumlah data: PASSED")


Total rows: 10000
Brand distribution:
brand
gojek    5000
grab     5000
Name: count, dtype: int64
Validasi jumlah data: PASSED


## 2) Preprocessing Teks dan Pelabelan Sentimen
Label dibentuk dari rating: 1-2 = negative, 3 = neutral, 4-5 = positive. Proses ini menjaga jumlah baris tetap sama.


In [3]:
URL_PATTERN = re.compile(r"https?://\S+|www\.\S+")
MENTION_PATTERN = re.compile(r"@[A-Za-z0-9_]+")
HASHTAG_PATTERN = re.compile(r"#[A-Za-z0-9_]+")
NON_ALNUM_PATTERN = re.compile(r"[^a-z0-9\s]")
WHITESPACE_PATTERN = re.compile(r"\s+")

def clean_text(text: object) -> str:
    value = str(text or "").lower()
    value = URL_PATTERN.sub(" ", value)
    value = MENTION_PATTERN.sub(" ", value)
    value = HASHTAG_PATTERN.sub(" ", value)
    value = NON_ALNUM_PATTERN.sub(" ", value)
    value = WHITESPACE_PATTERN.sub(" ", value).strip()
    return value or "tidak ada komentar"

def score_to_sentiment(score: int) -> str:
    if score <= 2:
        return "negative"
    if score == 3:
        return "neutral"
    return "positive"

processed_df = raw_df.copy()
rows_before = len(processed_df)

processed_df["username"] = processed_df["username"].fillna("unknown_user").astype(str).str.strip()
processed_df["content"] = processed_df["content"].fillna("").astype(str).str.strip()
processed_df.loc[processed_df["content"] == "", "content"] = "tidak ada komentar"
processed_df["score"] = pd.to_numeric(processed_df["score"], errors="coerce").fillna(3).astype(int)
processed_df["sentiment"] = processed_df["score"].apply(score_to_sentiment)
processed_df["clean_content"] = processed_df["content"].apply(clean_text)
processed_df["brand"] = processed_df["brand"].astype(str).str.lower().str.strip()
if "source" not in processed_df.columns:
    processed_df["source"] = "google_play"
if "app_id" not in processed_df.columns:
    processed_df["app_id"] = ""

ordered_columns = [
    "date",
    "username",
    "content",
    "clean_content",
    "score",
    "sentiment",
    "brand",
    "app_id",
    "source",
]
processed_df = processed_df[ordered_columns]

rows_after = len(processed_df)
print("Rows before preprocessing:", rows_before)
print("Rows after preprocessing :", rows_after)
assert rows_before == rows_after, "Jumlah data tidak boleh berkurang."

processed_df.head()


Rows before preprocessing: 10000
Rows after preprocessing : 10000


,date,username,content,clean_content,score,sentiment,brand,app_id,source
0,2026-04-11 12:24:23,Raffa Fathan,sangat membantu terima kasih karya anak bangsa,sangat membantu terima kasih karya anak bangsa,5,positive,gojek,com.gojek.app,google_play
1,2026-04-11 12:24:13,Muhammad zuki,Mau upgrade salah mulu photonya padahal uda be...,mau upgrade salah mulu photonya padahal uda be...,1,negative,gojek,com.gojek.app,google_play
2,2026-04-11 12:21:35,Atikah Tikah,"mobil nya bersih wangi,dan pelayanannya ramah",mobil nya bersih wangi dan pelayanannya ramah,4,positive,gojek,com.gojek.app,google_play
3,2026-04-11 12:17:04,Dwi Sandarti,"cepat, nyaman dan aman",cepat nyaman dan aman,5,positive,gojek,com.gojek.app,google_play
4,2026-04-11 11:50:29,TONI HERYAWAN JUARA GROUP,sangat membantu 👌🏼,sangat membantu,5,positive,gojek,com.gojek.app,google_play


In [4]:
processed_path = project_root / "data" / "processed" / "playstore_reviews_processed_latest.csv"
processed_path.parent.mkdir(parents=True, exist_ok=True)
processed_df.to_csv(processed_path, index=False)

print("Processed CSV saved:", processed_path)
print("Processed rows:", len(processed_df))
print("Sentiment distribution:")
display(processed_df["sentiment"].value_counts())
print("Brand distribution:")
display(processed_df["brand"].value_counts())


Processed CSV saved: C:\PT InfantAI Teknologi Nusantara\shabi\shabi_v2\data\processed\playstore_reviews_processed_latest.csv
Processed rows: 10000
Sentiment distribution:


sentiment
positive    5890
negative    3692
neutral      418
Name: count, dtype: int64

Brand distribution:


brand
gojek    5000
grab     5000
Name: count, dtype: int64

## 3) Pelatihan Model dan Evaluasi
Tiga skema machine learning yang dijalankan langsung di notebook:
1. SVM + TF-IDF (split 80/20)
2. Logistic Regression + TF-IDF (split 70/30)
3. Multinomial Naive Bayes + TF-IDF (split 80/20)


In [5]:
LABEL_ORDER = ["negative", "neutral", "positive"]

X = processed_df["clean_content"].astype(str)
y = processed_df["sentiment"].astype(str)

def run_experiment(name: str, split_label: str, test_size: float, model: Pipeline):
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=42,
        stratify=y,
    )

    model.fit(X_train, y_train)
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc = accuracy_score(y_test, y_test_pred)

    print(f"\n{name} ({split_label})")
    print(f"Train accuracy: {train_acc:.4f}")
    print(f"Test accuracy : {test_acc:.4f}")
    print(classification_report(y_test, y_test_pred, labels=LABEL_ORDER, zero_division=0))

    return {
        "experiment_name": name,
        "split": split_label,
        "train_accuracy": float(train_acc),
        "test_accuracy": float(test_acc),
        "model": model,
    }

experiments = [
    {
        "name": "experiment_1_svm_tfidf",
        "split": "80/20",
        "test_size": 0.2,
        "model": Pipeline(
            steps=[
                ("tfidf", TfidfVectorizer(ngram_range=(1, 2), max_features=60000, min_df=3, sublinear_tf=True)),
                ("classifier", LinearSVC(C=1.0)),
            ]
        ),
    },
    {
        "name": "experiment_2_logreg_tfidf",
        "split": "70/30",
        "test_size": 0.3,
        "model": Pipeline(
            steps=[
                ("tfidf", TfidfVectorizer(ngram_range=(1, 2), max_features=80000, min_df=2, sublinear_tf=True)),
                ("classifier", LogisticRegression(max_iter=2000)),
            ]
        ),
    },
    {
        "name": "experiment_3_nb_tfidf",
        "split": "80/20",
        "test_size": 0.2,
        "model": Pipeline(
            steps=[
                ("tfidf", TfidfVectorizer(ngram_range=(1, 2), max_features=60000, min_df=2, sublinear_tf=True)),
                ("classifier", MultinomialNB(alpha=0.5)),
            ]
        ),
    },
]

results = []
trained_models = {}

for cfg in experiments:
    output = run_experiment(
        name=cfg["name"],
        split_label=cfg["split"],
        test_size=cfg["test_size"],
        model=cfg["model"],
    )
    results.append({
        "experiment_name": output["experiment_name"],
        "split": output["split"],
        "train_accuracy": output["train_accuracy"],
        "test_accuracy": output["test_accuracy"],
    })
    trained_models[output["experiment_name"]] = output["model"]



experiment_1_svm_tfidf (80/20)
Train accuracy: 0.9768
Test accuracy : 0.8860
              precision    recall  f1-score   support

    negative       0.83      0.91      0.87       738
     neutral       0.21      0.04      0.06        84
    positive       0.93      0.93      0.93      1178

    accuracy                           0.89      2000
   macro avg       0.66      0.63      0.62      2000
weighted avg       0.86      0.89      0.87      2000




experiment_2_logreg_tfidf (70/30)
Train accuracy: 0.9273
Test accuracy : 0.8970
              precision    recall  f1-score   support

    negative       0.82      0.94      0.88      1108
     neutral       0.00      0.00      0.00       125
    positive       0.95      0.93      0.94      1767

    accuracy                           0.90      3000
   macro avg       0.59      0.62      0.61      3000
weighted avg       0.86      0.90      0.88      3000




experiment_3_nb_tfidf (80/20)
Train accuracy: 0.9175
Test accuracy : 0.8865
              precision    recall  f1-score   support

    negative       0.80      0.95      0.86       738
     neutral       0.00      0.00      0.00        84
    positive       0.96      0.91      0.93      1178

    accuracy                           0.89      2000
   macro avg       0.58      0.62      0.60      2000
weighted avg       0.86      0.89      0.87      2000



In [6]:
results_df = pd.DataFrame(results).sort_values("test_accuracy", ascending=False).reset_index(drop=True)
display(results_df)

passed_85 = int((results_df["test_accuracy"] >= 0.85).sum())
print("Jumlah eksperimen dengan test accuracy >= 85%:", passed_85)
assert passed_85 >= 1, "Minimal satu eksperimen harus mencapai akurasi test >= 85%."

models_dir = project_root / "models"
reports_dir = project_root / "reports"
models_dir.mkdir(parents=True, exist_ok=True)
reports_dir.mkdir(parents=True, exist_ok=True)

model_filename_map = {
    "experiment_1_svm_tfidf": "svm_tfidf_80_20.joblib",
    "experiment_2_logreg_tfidf": "logreg_tfidf_70_30.joblib",
    "experiment_3_nb_tfidf": "nb_tfidf_80_20.joblib",
}

for exp_name, model in trained_models.items():
    model_path = models_dir / model_filename_map[exp_name]
    joblib.dump(model, model_path)

metrics_path = reports_dir / "metrics_experiments.csv"
best_path = reports_dir / "best_experiment.json"

results_df.to_csv(metrics_path, index=False)
best_path.write_text(json.dumps(results_df.iloc[0].to_dict(), indent=2), encoding="utf-8")

print("Metrics saved:", metrics_path)
print("Best experiment saved:", best_path)


,experiment_name,split,train_accuracy,test_accuracy
0,experiment_2_logreg_tfidf,70/30,0.927286,0.8970
1,experiment_3_nb_tfidf,80/20,0.917500,0.8865
2,experiment_1_svm_tfidf,80/20,0.976750,0.8860


Jumlah eksperimen dengan test accuracy >= 85%: 3


Metrics saved: C:\PT InfantAI Teknologi Nusantara\shabi\shabi_v2\reports\metrics_experiments.csv
Best experiment saved: C:\PT InfantAI Teknologi Nusantara\shabi\shabi_v2\reports\best_experiment.json


## 4) Inference Output Kategorikal
Prediksi menghasilkan label kelas: negative, neutral, atau positive.


In [7]:
best_experiment_name = results_df.iloc[0]["experiment_name"]
best_model = trained_models[best_experiment_name]

sample_texts = [
    "Aplikasi bagus, driver cepat, saya puas.",
    "Aplikasinya sering error dan susah dipakai.",
    "Biasa saja, tidak terlalu bagus dan tidak buruk.",
]

sample_clean = [clean_text(text) for text in sample_texts]
sample_preds = best_model.predict(sample_clean)

print("Best model:", best_experiment_name)
print("Inference output (categorical labels):")
for idx, (text, pred) in enumerate(zip(sample_texts, sample_preds), start=1):
    print(f"{idx}. [{pred}] {text}")


Best model: experiment_2_logreg_tfidf
Inference output (categorical labels):
1. [positive] Aplikasi bagus, driver cepat, saya puas.
2. [negative] Aplikasinya sering error dan susah dipakai.
3. [negative] Biasa saja, tidak terlalu bagus dan tidak buruk.
